# providers

> Value-string provider registry

In [ ]:
#| default_exp providers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import pathlib, datetime, enum, uuid
import numpy as np, pandas as pd
from paar.providers import str_provider, register_str_provider, provided_str, value_str

class Color(enum.Enum): RED=1; GREEN=2
class MyInt(enum.IntEnum): A=1

def test_path():        assert value_str(pathlib.Path('/tmp/x'))==str(pathlib.Path('/tmp/x'))
def test_datetime():    assert value_str(datetime.datetime(2020,1,2,3,4,5))=='2020-01-02T03:04:05'
def test_date():        assert value_str(datetime.date(2020,1,2))=='2020-01-02'
def test_enum():        assert value_str(Color.RED)=='Color.RED'
def test_intenum_sub(): assert value_str(MyInt.A)=='MyInt.A'
def test_uuid():
    u=uuid.UUID('12345678-1234-5678-1234-567812345678'); assert value_str(u)==str(u)
def test_bytes_short(): assert value_str(b'abc')=='616263 (3 bytes)'
def test_bytes_long():
    s=value_str(bytes(range(20))); assert s.endswith('(20 bytes)') and '…' in s
def test_ndarray():     assert value_str(np.arange(6).reshape(2,3))=='ndarray shape=(2, 3) int64'
def test_dataframe():   assert value_str(pd.DataFrame({'a':[1],'b':[2]}))=='DataFrame [1×2]'
def test_series():      assert value_str(pd.Series([1,2,3]))=='Series [3] int64'
def test_unregistered_falls_back(): assert value_str(123)=='123'
def test_latest_registered_wins():
    class Widget: pass
    register_str_provider(Widget, lambda w: 'A')
    register_str_provider(Widget, lambda w: 'B')
    assert value_str(Widget())=='B'
def test_throwing_provider_falls_back():
    class Boom: pass
    register_str_provider(Boom, lambda x: 1/0)
    assert value_str(Boom()).startswith('<')   # safe_repr of the Boom instance
def test_bytearray():
    assert value_str(bytearray(b'abc'))=='616263 (3 bytes)'
def test_throwing_falls_through_to_earlier():
    class W: pass
    register_str_provider(W, lambda x: 'ok')     # earlier
    register_str_provider(W, lambda x: 1/0)      # latest, throws -> should be skipped
    assert value_str(W())=='ok'                  # falls through to the earlier provider
for t in [test_path,test_datetime,test_date,test_enum,test_intenum_sub,test_uuid,
          test_bytes_short,test_bytes_long,test_ndarray,test_dataframe,test_series,
          test_unregistered_falls_back,test_latest_registered_wins,test_throwing_provider_falls_back,
          test_bytearray,test_throwing_falls_through_to_earlier]: t()

In [ ]:
#| export
import pathlib, datetime, enum, uuid
from paar.reprs import safe_repr
try: import numpy as np
except Exception: np = None
try: import pandas as pd
except Exception: pd = None

_PROVIDERS = []   # list[(type, fn)]

def register_str_provider(typ, fn):
    "Register fn(value)->str as the display string for instances of typ (and subclasses)."
    _PROVIDERS.append((typ, fn))

def str_provider(typ):
    "Decorator: @str_provider(T) registers the decorated fn as T's value-string provider."
    def deco(fn): register_str_provider(typ, fn); return fn
    return deco

def provided_str(v):
    "Custom display string for v (latest-registered isinstance match), or None."
    for typ, fn in reversed(_PROVIDERS):
        try:
            if isinstance(v, typ): return fn(v)
        except Exception:
            continue
    return None

def value_str(v):
    "Display string for v: a registered provider if any, else safe_repr(v)."
    s = provided_str(v)
    return s if s is not None else safe_repr(v)

def _bytes_str(b):
    h = bytes(b[:8]).hex()
    return f'{h}… ({len(b)} bytes)' if len(b) > 8 else f'{h} ({len(b)} bytes)'

register_str_provider(pathlib.Path, lambda p: str(p))
register_str_provider(datetime.date, lambda d: d.isoformat())
register_str_provider(datetime.time, lambda t: t.isoformat())
register_str_provider(datetime.datetime, lambda d: d.isoformat())
register_str_provider(enum.Enum, lambda e: f'{type(e).__name__}.{e.name}')
register_str_provider(uuid.UUID, lambda u: str(u))
register_str_provider(bytes, _bytes_str)
register_str_provider(bytearray, _bytes_str)
if np is not None:
    register_str_provider(np.ndarray, lambda a: f'ndarray shape={tuple(a.shape)} {a.dtype}')
if pd is not None:
    register_str_provider(pd.DataFrame, lambda d: f'DataFrame [{d.shape[0]}×{d.shape[1]}]')
    register_str_provider(pd.Series, lambda s: f'Series [{s.shape[0]}] {s.dtype}')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()